# Drone actuator fault: how many seconds until the alarm?

A drone's commanded rate and measured rate should agree. When an actuator loses authority the gap grows. This notebook feeds a synthetic 100 Hz flight to a CUSUM detector, one sample at a time, and measures the detection delay and the false alarms per hour. The flights are **synthetic**, generated by `pipelines.drones.io.synthetic_flight`; the ALFA real-flight benchmark lives in `benchmarks/drones/alfa_actuator.py`.

In [ ]:
!pip install -q dense-armor

## 1. One flight with the fault

In [ ]:
from pipelines.drones.io import synthetic_flight
from pipelines.drones.actuator_fault import run

stream = list(synthetic_flight(6000, seed=1, fault='actuator_loss'))
r = run(stream)
print('fault at sample', r.fault_index, '| episodes', r.n_episodes, '| delay s', r.max_delay_s)

The fault starts at sample 3600 (36 s). The detector raises one alarm episode, 0.02 s later.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

t = np.arange(r.n_samples) / 100
fig, ax = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
ax[0].plot(t, r.mahalanobis_scores, lw=0.6)
ax[0].set_ylabel('Mahalanobis score')
ax[1].plot(t, r.cusum_alarms, lw=0.8)
ax[1].set_ylabel('CUSUM alarm')
for a in ax:
    a.axvline(r.fault_index / 100, color='r', ls='--')
ax[1].set_xlabel('seconds')
plt.show()

## 2. Many seeds: delay and false alarms

In [ ]:
import numpy as np

delays, fa, sec = [], 0, 0.0
for s in range(20):
    q = run(list(synthetic_flight(6000, seed=s, fault='actuator_loss')))
    delays += q.delays_s
    fa += q.false_alarms
    sec += q.normal_seconds
print('episodes after the fault', len(delays), 'in 20 flights | median delay s', round(float(np.median(delays)), 3))
print('false alarms per hour', round(fa * 3600 / sec, 2))

## 3. The threshold trap

`h` is the CUSUM threshold. A textbook `h=5` looks reasonable, yet on a stable stream of 1000 samples the average run length without change is 469, so about 88% of such streams raise a false alarm. Raising `h` removes them. On this synthetic fault the delay stays small up to h=20.

In [ ]:
rows = []
for h in (5.0, 10.0, 20.0):
    fa, sec, d = 0, 0.0, []
    for s in range(20):
        q = run(list(synthetic_flight(6000, seed=s, fault='actuator_loss')), cusum_kwargs={'h': h})
        fa += q.false_alarms
        sec += q.normal_seconds
        d += q.delays_s
    rows.append((h, round(fa * 3600 / sec, 1), round(float(np.median(d)), 3) if d else None))
for h, f, d in rows:
    print('h', h, '| false alarms/hour', f, '| median delay s', d)

Everything above is `synthetic`. Source and docs: https://github.com/tatopenn-cell/Dense-Armor